<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Physical Duckiedrone Access

Secure Shell (SSH) lets you open an encrypted remote shell on a Duckiedrone from your base station. You can inspect files and running programs without attaching a keyboard or screen to the Duckiedrone.

In this notebook, you will connect to your Duckiedrone, verify where commands run, and return to the base station. An optional final step configures a personal key for future logins.

SSH gives you the permissions of the account you log into, not automatically unrestricted access. Depending on the Duckiedrone's configuration, you can authenticate with a password or a personal key; creating a key pair is not always required before the first connection.

## Verify the intended Duckiedrone first

SSH grants remote shell access. From a base-station terminal on a network you own or are authorized to use, connect only to a Duckiedrone you own or are authorized to access.

First, make sure your base station and Duckiedrone are on the intended network. See the Duckietown Manual for [how to configure the network on a Duckiedrone](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/20-first-connection.html).

Find the Duckiedrone's assigned name in its setup record or ask the device owner.

From the base-station terminal, make a bounded reachability check:

```bash
ping -c 3 DUCKIEDRONE_NAME.local
```

A successful `ping` shows that the Duckiedrone replied to an Internet Control Message Protocol (ICMP) echo request. If it fails, check the name and network path, but remember that ICMP can be blocked even when SSH is available.

## Open an authorized remote session

From the same base-station terminal, run:

```bash
ssh duckie@DUCKIEDRONE_NAME.local
```

On the first connection, SSH displays a host-key fingerprint, a short identifier for the Duckiedrone's cryptographic identity. For illustration, the following outputs use the example physical Duckiedrone `amelia` and its recorded address `192.168.1.201`; your Duckiedrone will show its own name, address, and fingerprint:

```shell
The authenticity of host 'amelia.local (192.168.1.201)' can't be established.
ED25519 key fingerprint is SHA256:...
Are you sure you want to continue connecting (yes/no/[fingerprint])?
```

The fingerprint is abbreviated here. Compare the complete fingerprint shown in your terminal with a trusted setup record. Enter `yes` only when they match. If you have no trusted fingerprint, obtain one from the device owner before accepting the key.

SSH normally saves the accepted host key in your base station's `~/.ssh/known_hosts` file. This lets it recognize the server on later connections. An unexpected change may follow reinstallation, but needs verification before you accept it. See [OpenSSH's host-key checking documentation](https://man.openbsd.org/ssh_config.5#StrictHostKeyChecking) for additional detail.

You may then be prompted for the `duckie` account password on the Duckiedrone:

```shell
duckie@amelia.local's password:
```

Enter the password configured for that Duckiedrone account. No characters or asterisks appear while you type; that is normal. Press `Enter` when finished. If an authorized personal key is already configured, you may not receive a Duckiedrone-password prompt.

After the session opens, the prompt might look like this:

```shell
duckie@amelia:~ $
```

### Try it: verify and close the session

Confirm where you are before exploring the Duckiedrone:

```bash
hostname
whoami
pwd
```

On the example Duckiedrone `amelia`, these commands might print:

```shell
amelia
duckie
/home/duckie
```

These results identify the computer, account, and current directory. From this point, commands entered in this terminal operate on the Duckiedrone. For example, `ls` lists files on the Duckiedrone, and `localhost` refers to the Duckiedrone's network context.

[Figure 1](#figure-1) shows how to return to the base station.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    base-station shell
      |
      | SSH
      v
    Duckiedrone shell
      |
      | exit
      v
    base-station shell
  </pre>
  <figcaption>
    Figure 1: SSH opens a Duckiedrone shell; <code>exit</code> returns to the base station.
  </figcaption>
</figure>

To close the remote session and return to the base station, type `exit`. Run `hostname` again to confirm you are back on your base station. Leaving the remote session does not shut down the Duckiedrone. [OpenSSH's ssh manual](https://man.openbsd.org/ssh.1) describes remote sessions and authentication.

<details>
<summary>Check your result</summary>

The hostname should change when you enter and leave the remote shell. In the Duckiedrone shell, `whoami` identifies the account you used to log in; exiting closes only the SSH session.

</details>

### If login fails

Several problems can prevent an SSH connection. [Table 1](#table-1) lists common symptoms.

<table id="table-1" class="lx-table">
  <caption>Table 1: SSH errors and what to investigate.</caption>
  <thead>
    <tr><th>Message</th><th>What to investigate</th></tr>
  </thead>
  <tbody>
    <tr><td><code>Could not resolve hostname</code></td><td>The hostname and name-resolution setup</td></tr>
    <tr><td><code>Connection refused</code> or a timeout</td><td>The network path, SSH service, and access rules</td></tr>
    <tr><td><code>Permission denied</code></td><td>The username and accepted password or key</td></tr>
    <tr><td>Host-key change warning</td><td>Whether the device's identity changed as expected</td></tr>
  </tbody>
</table>

For example, `Permission denied` after password entry means you reached an SSH server but authentication failed. Repeating network discovery will not correct the account credentials.

## Install one personal public key

For repeated connections, a personal key can replace entering the Duckiedrone's account password. This is separate from the host key checked above, as [Table 2](#table-2) shows.

<table id="table-2" class="lx-table">
  <caption>Table 2: SSH host and personal key roles.</caption>
  <thead>
    <tr><th>Key</th><th>Purpose</th></tr>
  </thead>
  <tbody>
    <tr><td>Duckiedrone's host key</td><td>Identifies the SSH server</td></tr>
    <tr><td>Your personal key pair</td><td>Authenticates you to the Duckiedrone account</td></tr>
  </tbody>
</table>

Your __private key__ stays on the base station. Its matching __public key__ can be installed on the Duckiedrone.

This step is optional and changes the account's future access configuration. Use it when your device setup permits personal keys and a password login has already worked.

Back in the __base-station terminal__, check for your existing public key:

```bash
ls ~/.ssh/id_ed25519.pub
```

If it is missing, follow the device's key-management guidance or skip this optional step. If your personal key uses another filename, use that path instead.

Once available, install the public key on the Duckiedrone:

```bash
ssh-copy-id -i ~/.ssh/id_ed25519.pub "duckie@DUCKIEDRONE_NAME.local"
```

This command authenticates to the Duckiedrone and adds the selected public key to the account's authorized keys. It normally requests the Duckiedrone password if the key is not already accepted. See the [`ssh-copy-id` manual](https://man7.org/linux/man-pages/man1/ssh-copy-id.1.html).

If you set a passphrase when creating the key, it protects your local private key; it is separate from the Duckiedrone account password. Never copy the private key to the Duckiedrone or include it in shared output.

Reconnect using the SSH command in [Open an authorized remote session](#open-an-authorized-remote-session). Run `hostname` to confirm the destination, then `exit`. If SSH asks for the Duckiedrone account password rather than a local key passphrase, key-based login has not been verified.

## Further reading

The OpenSSH [`ssh` client manual](https://man.openbsd.org/ssh.1) and [`ssh-copy-id` manual](https://man7.org/linux/man-pages/man1/ssh-copy-id.1.html) provide additional connection and key-management detail.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
